In [1]:
# --- Dependencies ---
!pip install torch torchvision torchaudio numpy pandas matplotlib scikit-learn tqdm pillow opencv-python mediapipe

# Imports & Setups

In [2]:
import os, json, random
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from tqdm import tqdm
from collections import Counter
from pathlib import Path
import cv2
import mediapipe as mp
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image, UnidentifiedImageError

In [3]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}\n")

Using device: cuda



# Configuration

In [4]:
INPUT_JSON = Path("../data/labeled_images/_annotations.coco.json")
CLEAN_JSON = INPUT_JSON.with_name("_annotations.coco_cleaned.json")
data_root = INPUT_JSON.parent

MODEL2_DIR = Path("model2_multitask")
MODEL2_DIR.mkdir(exist_ok=True)

CFG = {
    "img_size": 224,
    "batch_size": 32,
    "test_size": 0.15,
    "val_size": 0.1765,
    "context_scale": 1.20,
    "epochs": 15,
    "lr_backbone": 1e-4,
    "lr_heads": 1e-3,
    "weight_decay": 1e-4,
    "w_drowsy": 1.0,
    "w_eye": 0.5,
    "w_mouth": 0.5,
    "fusion_w_base": 0.6,
    "fusion_w_eye": 0.25,
    "fusion_w_mouth": 0.15,
}

print("Configuration:")
for k, v in CFG.items():
    print(f"  {k}: {v}")
print()

Configuration:
  img_size: 224
  batch_size: 32
  test_size: 0.15
  val_size: 0.1765
  context_scale: 1.2
  epochs: 15
  lr_backbone: 0.0001
  lr_heads: 0.001
  weight_decay: 0.0001
  w_drowsy: 1.0
  w_eye: 0.5
  w_mouth: 0.5
  fusion_w_base: 0.6
  fusion_w_eye: 0.25
  fusion_w_mouth: 0.15



# Data Cleaning

In [5]:
def read_json(path):
    with open(path, "r") as f:
        return json.load(f)

def write_json(obj, path):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)

# Check if cleaned file exists, if not, clean the raw data
if not CLEAN_JSON.exists():
    print("Cleaned annotations not found. Running data cleaning...")
    
    coco = read_json(INPUT_JSON)
    
    def ensure_cat(coco, name, supercat="person-state"):
        for c in coco.get("categories", []):
            if c["name"].lower() == name.lower():
                c["name"] = name
                return c["id"]
        new_id = max([c["id"] for c in coco.get("categories", [])] + [0]) + 1
        coco.setdefault("categories", []).append({"id": new_id, "name": name, "supercategory": supercat})
        return new_id
    
    # Get face category IDs
    face_ids = {c["id"] for c in coco.get("categories", []) if c.get("name", "").lower() == "face"}
    alert_id = ensure_cat(coco, "alert")
    drowsy_id = ensure_cat(coco, "drowsy")
    
    # Build primary labels from image tags
    primary = {}
    drop_ids = set()
    
    for img in coco.get("images", []):
        tags = [t.lower() for t in img.get("extra", {}).get("user_tags", []) if isinstance(t, str)]
        tagset = set(tags)
        
        # If both alert and drowsy, keep only alert
        if "alert" in tagset and "drowsy" in tagset:
            tagset = {"alert"}
        
        # Drop images with only uncertain
        if tagset == {"uncertain"}:
            drop_ids.add(img["id"])
            continue
        
        # Determine primary label
        if "alert" in tagset:
            primary[img["id"]] = "alert"
            img.setdefault("extra", {})["user_tags"] = ["alert"]
        elif "drowsy" in tagset:
            primary[img["id"]] = "drowsy"
            img.setdefault("extra", {})["user_tags"] = ["drowsy"]
        else:
            primary[img["id"]] = None
    
    # Remove dropped images and annotations
    if drop_ids:
        coco["images"] = [im for im in coco["images"] if im["id"] not in drop_ids]
        coco["annotations"] = [a for a in coco["annotations"] if a["image_id"] not in drop_ids]
    
    # Convert face annotations to alert/drowsy
    changed = 0
    for ann in coco["annotations"]:
        if ann.get("category_id") in face_ids:
            label = primary.get(ann["image_id"])
            if label == "alert":
                ann["category_id"] = alert_id
                changed += 1
            elif label == "drowsy":
                ann["category_id"] = drowsy_id
                changed += 1
    
    write_json(coco, CLEAN_JSON)
    print(f"  Converted {changed} face annotations")
    print(f"  Dropped {len(drop_ids)} uncertain images")
    print(f"  Saved to: {CLEAN_JSON}\n")
else:
    print("Using existing cleaned annotations\n")

Using existing cleaned annotations



# Load Annotated Data

In [6]:
coco = read_json(CLEAN_JSON)
print(f"Loaded {len(coco.get('images', []))} images")
print(f"Loaded {len(coco.get('annotations', []))} annotations")
print(f"Categories: {[c['name'] for c in coco.get('categories', [])]}\n")

# Build image index
def build_img_index(coco, data_root):
    idx = {}
    for img in coco.get("images", []):
        rel = Path(img.get("file_name", ""))
        p = (data_root / rel).resolve()
        if p.exists():
            tags = [t.lower() for t in img.get("extra", {}).get("user_tags", [])]
            idx[img["id"]] = {
                "path": str(p),
                "width": img.get("width"),
                "height": img.get("height"),
                "tags": tags
            }
    return idx

img_index = build_img_index(coco, data_root)
print(f"Indexed {len(img_index)} valid images\n")

# Build category name to ID mapping
name_to_id = {c["name"].lower(): c["id"] for c in coco.get("categories", [])}
ALERT_ID = name_to_id.get("alert")
DROWSY_ID = name_to_id.get("drowsy")
EYES_CLOSE_ID = name_to_id.get("eyes_close")
EYES_OPEN_ID = name_to_id.get("eyes_open")
MOUTH_CLOSE_ID = name_to_id.get("mouth_close")
YAWN_ID = name_to_id.get("yawn")

print(f"Category IDs:")
print(f"  Alert={ALERT_ID}, Drowsy={DROWSY_ID}")
print(f"  Eyes: Close={EYES_CLOSE_ID}, Open={EYES_OPEN_ID}")
print(f"  Mouth: Close={MOUTH_CLOSE_ID}, Yawn={YAWN_ID}\n")

# Build annotation lookup by image_id
img_to_anns = {}
for ann in coco.get("annotations", []):
    img_id = ann.get("image_id")
    if img_id not in img_to_anns:
        img_to_anns[img_id] = []
    img_to_anns[img_id].append(ann)

def extract_multitask_labels_from_anns(img_id, img_to_anns):
    """Extract eye and mouth labels from annotations for this image."""
    anns = img_to_anns.get(img_id, [])
    
    y_eye = 2  # default: uncertain
    y_mouth = 2  # default: uncertain
    
    for ann in anns:
        cid = ann.get("category_id")
        if cid == EYES_CLOSE_ID:
            y_eye = 0
        elif cid == EYES_OPEN_ID:
            y_eye = 1
        elif cid == MOUTH_CLOSE_ID:
            y_mouth = 0
        elif cid == YAWN_ID:
            y_mouth = 1
    
    return y_eye, y_mouth

# Build samples: (img_path, bbox, y_drowsy, y_eye, y_mouth)
# Only use face/alert/drowsy bboxes as the main crop
samples = []
for ann in coco.get("annotations", []):
    cid = ann.get("category_id")
    if cid not in (ALERT_ID, DROWSY_ID):
        continue
    
    img_id = ann.get("image_id")
    if img_id not in img_index:
        continue
    
    img_info = img_index[img_id]
    bbox = ann.get("bbox")
    if not bbox or len(bbox) != 4:
        continue
    
    # Primary label
    y_drowsy = 1 if cid == ALERT_ID else 0
    
    # Get eye/mouth from other annotations on same image
    y_eye, y_mouth = extract_multitask_labels_from_anns(img_id, img_to_anns)
    
    samples.append((img_info["path"], tuple(bbox), y_drowsy, y_eye, y_mouth))

print(f"Extracted {len(samples)} samples")
print(f"Drowsy dist: {dict(Counter([s[2] for s in samples]))}")
print(f"Eye dist:    {dict(Counter([s[3] for s in samples]))}")
print(f"Mouth dist:  {dict(Counter([s[4] for s in samples]))}")
print("(0=Drowsy/Closed, 1=Alert/Open/Yawn, 2=Uncertain)\n")

Loaded 3166 images
Loaded 9491 annotations
Categories: ['face-eyes-mouth', 'eyes_close', 'eyes_open', 'face', 'mouth_close', 'uncertain', 'yawn', 'alert', 'drowsy']

Indexed 3166 valid images

Category IDs:
  Alert=7, Drowsy=8
  Eyes: Close=1, Open=2
  Mouth: Close=4, Yawn=6

Extracted 3159 samples
Drowsy dist: {0: 1560, 1: 1599}
Eye dist:    {0: 752, 1: 2236, 2: 171}
Mouth dist:  {0: 2433, 1: 714, 2: 12}
(0=Drowsy/Closed, 1=Alert/Open/Yawn, 2=Uncertain)



# Facial Landmark Extractor

In [7]:
class LandmarkExtractor:
    """Extract facial landmarks using MediaPipe."""
    
    def __init__(self, confidence=0.5):
        self.mp_face_mesh = mp.solutions.face_mesh
        self.face_mesh = self.mp_face_mesh.FaceMesh(
            static_image_mode=True,
            max_num_faces=1,
            min_detection_confidence=confidence
        )
        
        # Key landmark indices
        self.left_eye = [33, 160, 158, 133, 153, 144]
        self.right_eye = [362, 385, 387, 263, 373, 380]
        self.mouth = [61, 291, 0, 17, 39, 269]
    
    def extract(self, img_pil):
        """Extract 18 landmark features (6 per eye, 6 for mouth)."""
        img_np = np.array(img_pil)
        img_rgb = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR)
        
        results = self.face_mesh.process(img_rgb)
        
        if not results.multi_face_landmarks:
            return np.zeros(18, dtype=np.float32)
        
        landmarks = results.multi_face_landmarks[0].landmark
        features = []
        
        for idx in self.left_eye:
            features.append(landmarks[idx].y)
        for idx in self.right_eye:
            features.append(landmarks[idx].y)
        for idx in self.mouth:
            features.append(landmarks[idx].y)
        
        return np.array(features, dtype=np.float32)

landmark_extractor = LandmarkExtractor()
print("Landmark extractor initialized\n")

Landmark extractor initialized



# Dataset and Dataloader

In [8]:
class MultitaskCropDataset(Dataset):
    """Dataset returning: image, multi-task labels, landmarks."""
    
    def __init__(self, samples, transform, landmark_extractor, 
                 context_scale=1.20, min_size=8):
        self.samples = samples
        self.transform = transform
        self.landmark_extractor = landmark_extractor
        self.context_scale = context_scale
        self.min_size = min_size
    
    def __len__(self):
        return len(self.samples)
    
    def _expand_clip(self, x, y, w, h, W, H):
        cx, cy = x + w/2, y + h/2
        w2, h2 = w * self.context_scale, h * self.context_scale
        x1 = int(cx - w2/2); y1 = int(cy - h2/2)
        x2 = int(cx + w2/2); y2 = int(cy + h2/2)
        x1 = max(0, min(x1, W-1)); y1 = max(0, min(y1, H-1))
        x2 = max(1, min(x2, W)); y2 = max(1, min(y2, H))
        if x2 - x1 < self.min_size: x2 = min(W, x1 + self.min_size)
        if y2 - y1 < self.min_size: y2 = min(H, y1 + self.min_size)
        return x1, y1, x2, y2
    
    def __getitem__(self, idx):
        img_path, bbox, y_drowsy, y_eye, y_mouth = self.samples[idx]
        
        try:
            img = Image.open(img_path).convert("RGB")
        except Exception as e:
            raise RuntimeError(f"Failed to open {img_path}") from e
        
        W, H = img.size
        x, y0, w, h = bbox
        x1, y1, x2, y2 = self._expand_clip(x, y0, w, h, W, H)
        crop = img.crop((x1, y1, x2, y2))
        
        # Extract landmarks before transform
        landmarks = self.landmark_extractor.extract(crop)
        
        # Transform image
        crop_tensor = self.transform(crop)
        
        return crop_tensor, y_drowsy, y_eye, y_mouth, landmarks

# Transforms
train_tfm = transforms.Compose([
    transforms.RandomResizedCrop(CFG["img_size"], scale=(0.9, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=7),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

eval_tfm = transforms.Compose([
    transforms.Resize((CFG["img_size"], CFG["img_size"])),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# Split data
all_idx = np.arange(len(samples))
y_all = np.array([samples[i][2] for i in all_idx])

idx_trv, idx_te = train_test_split(
    all_idx, test_size=CFG["test_size"], random_state=42, stratify=y_all
)
y_trv = y_all[idx_trv]

val_ratio = CFG["val_size"] / (1.0 - CFG["test_size"])
idx_tr, idx_va = train_test_split(
    idx_trv, test_size=val_ratio, random_state=42, stratify=y_trv
)

train_samples = [samples[i] for i in idx_tr]
val_samples = [samples[i] for i in idx_va]
test_samples = [samples[i] for i in idx_te]

print(f"Data split: Train={len(train_samples)}, Val={len(val_samples)}, Test={len(test_samples)}\n")

# Create dataloaders
train_loader = DataLoader(
    MultitaskCropDataset(train_samples, train_tfm, landmark_extractor, CFG["context_scale"]),
    batch_size=CFG["batch_size"], shuffle=True, num_workers=0
)

val_loader = DataLoader(
    MultitaskCropDataset(val_samples, eval_tfm, landmark_extractor, CFG["context_scale"]),
    batch_size=CFG["batch_size"], shuffle=False, num_workers=0
)

test_loader = DataLoader(
    MultitaskCropDataset(test_samples, eval_tfm, landmark_extractor, CFG["context_scale"]),
    batch_size=CFG["batch_size"], shuffle=False, num_workers=0
)

Data split: Train=2127, Val=558, Test=474



# Model Architecture

In [9]:
class MultitaskDrowsinessModel(nn.Module):
    """Multi-task model: ResNet18 + Landmarks + 3 heads."""
    
    def __init__(self, landmark_dim=18):
        super().__init__()
        
        # Backbone
        resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.backbone = nn.Sequential(*list(resnet.children())[:-1])
        backbone_dim = 512
        
        # Landmark processor
        self.landmark_fc = nn.Sequential(
            nn.Linear(landmark_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU()
        )
        
        combined_dim = backbone_dim + 32
        
        # Three heads
        self.drowsy_head = nn.Sequential(
            nn.Linear(combined_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, 2)
        )
        
        self.eye_head = nn.Sequential(
            nn.Linear(combined_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 3)
        )
        
        self.mouth_head = nn.Sequential(
            nn.Linear(combined_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 3)
        )
    
    def forward(self, x, landmarks):
        cnn_feat = self.backbone(x).flatten(1)
        lm_feat = self.landmark_fc(landmarks)
        combined = torch.cat([cnn_feat, lm_feat], dim=1)
        
        drowsy_logits = self.drowsy_head(combined)
        eye_logits = self.eye_head(combined)
        mouth_logits = self.mouth_head(combined)
        
        return drowsy_logits, eye_logits, mouth_logits

def build_model():
    model = MultitaskDrowsinessModel()
    # Freeze backbone except last 2 blocks
    for param in model.backbone.parameters():
        param.requires_grad = False
    for param in model.backbone[6].parameters():  # layer3
        param.requires_grad = True
    for param in model.backbone[7].parameters():  # layer4
        param.requires_grad = True
    return model

print("Model architecture defined\n")

Model architecture defined



# Multi-Task Loss

In [10]:
class MultitaskLoss(nn.Module):
    """Weighted multi-task loss."""
    
    def __init__(self, w_drowsy=1.0, w_eye=0.5, w_mouth=0.5):
        super().__init__()
        self.w_drowsy = w_drowsy
        self.w_eye = w_eye
        self.w_mouth = w_mouth
        self.ce_drowsy = nn.CrossEntropyLoss()
        self.ce_eye = nn.CrossEntropyLoss(ignore_index=2)
        self.ce_mouth = nn.CrossEntropyLoss(ignore_index=2)
    
    def forward(self, drowsy_logits, eye_logits, mouth_logits, 
                y_drowsy, y_eye, y_mouth):
        loss_d = self.ce_drowsy(drowsy_logits, y_drowsy)
        loss_e = self.ce_eye(eye_logits, y_eye)
        loss_m = self.ce_mouth(mouth_logits, y_mouth)
        
        total = self.w_drowsy * loss_d + self.w_eye * loss_e + self.w_mouth * loss_m
        return total, loss_d, loss_e, loss_m

criterion = MultitaskLoss(CFG["w_drowsy"], CFG["w_eye"], CFG["w_mouth"])
print("Multi-task loss initialized\n")

Multi-task loss initialized



# Training Functions

In [11]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    
    pbar = tqdm(loader, desc="Training")
    for x, y_d, y_e, y_m, lm in pbar:
        x, y_d, y_e, y_m, lm = x.to(device), y_d.to(device), y_e.to(device), y_m.to(device), lm.to(device)
        
        optimizer.zero_grad()
        logits_d, logits_e, logits_m = model(x, lm)
        loss, _, _, _ = criterion(logits_d, logits_e, logits_m, y_d, y_e, y_m)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * x.size(0)
        correct += (logits_d.argmax(1) == y_d).sum().item()
        total += x.size(0)
        pbar.set_postfix(loss=f"{total_loss/total:.4f}", acc=f"{correct/total:.4f}")
    
    return total_loss/total, correct/total

def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = {'d': [], 'e': [], 'm': []}, {'d': [], 'e': [], 'm': []}
    
    with torch.no_grad():
        for x, y_d, y_e, y_m, lm in tqdm(loader, desc="Evaluating"):
            x, y_d, y_e, y_m, lm = x.to(device), y_d.to(device), y_e.to(device), y_m.to(device), lm.to(device)
            
            logits_d, logits_e, logits_m = model(x, lm)
            loss, _, _, _ = criterion(logits_d, logits_e, logits_m, y_d, y_e, y_m)
            
            total_loss += loss.item() * x.size(0)
            correct += (logits_d.argmax(1) == y_d).sum().item()
            total += x.size(0)
            
            all_preds['d'].append(logits_d.argmax(1).cpu())
            all_preds['e'].append(logits_e.argmax(1).cpu())
            all_preds['m'].append(logits_m.argmax(1).cpu())
            all_labels['d'].append(y_d.cpu())
            all_labels['e'].append(y_e.cpu())
            all_labels['m'].append(y_m.cpu())
    
    preds = {k: torch.cat(v).numpy() for k, v in all_preds.items()}
    labels = {k: torch.cat(v).numpy() for k, v in all_labels.items()}
    
    return total_loss/total, correct/total, preds, labels

# Training Loop

In [ ]:
print("="*60)
print("Starting Training")
print("="*60)

model = build_model().to(device)

optimizer = optim.AdamW([
    {'params': model.backbone.parameters(), 'lr': CFG["lr_backbone"]},
    {'params': model.landmark_fc.parameters(), 'lr': CFG["lr_heads"]},
    {'params': model.drowsy_head.parameters(), 'lr': CFG["lr_heads"]},
    {'params': model.eye_head.parameters(), 'lr': CFG["lr_heads"]},
    {'params': model.mouth_head.parameters(), 'lr': CFG["lr_heads"]},
], weight_decay=CFG["weight_decay"])

scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc, best_epoch, best_state = 0.0, 0, None

for epoch in range(1, CFG["epochs"] + 1):
    print(f"\nEpoch {epoch}/{CFG['epochs']}")
    
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc, val_preds, val_labels = eval_epoch(model, val_loader, criterion, device)
    
    scheduler.step(val_acc)
    
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    print(f"Train: Loss={train_loss:.4f}, Acc={train_acc:.4f}")
    print(f"Val:   Loss={val_loss:.4f}, Acc={val_acc:.4f}")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch
        best_state = model.state_dict().copy()
        print(f"*** New best! Val Acc: {val_acc:.4f} ***")

print(f"\nTraining complete! Best Val Acc: {best_val_acc:.4f} at epoch {best_epoch}\n")

STARTING TRAINING

Epoch 1/15


Evaluating: 100%|██████████| 18/18 [00:06<00:00,  2.84it/s]


Train: Loss=0.6745, Acc=0.8340
Val:   Loss=0.3578, Acc=0.9301
*** New best! Val Acc: 0.9301 ***

Epoch 2/15


Training:  21%|██        | 14/67 [00:05<00:19,  2.75it/s, acc=0.9420, loss=0.3442]

# Test Evaluation of Base Model

In [ ]:
print("="*60)
print("EVALUATING ON TEST SET")
print("="*60)

model.load_state_dict(best_state)
test_loss, test_acc_base, test_preds, test_labels = eval_epoch(model, test_loader, criterion, device)

print(f"\nTest Results (Base Model):")
print(f"  Loss: {test_loss:.4f}")
print(f"  Accuracy: {test_acc_base:.4f}\n")

print("=== Drowsiness Classification ===")
print(classification_report(test_labels['d'], test_preds['d'], 
                          target_names=['Drowsy', 'Alert'], digits=4))

# Late Funsion

In [ ]:
def late_fusion_predict(drowsy_probs, eye_probs, mouth_probs, 
                       w_base=0.6, w_eye=0.25, w_mouth=0.15):
    """Weighted fusion of predictions."""
    base_alert = drowsy_probs[:, 1:2]
    eye_alert = eye_probs[:, 1:2]
    mouth_alert = 1.0 - mouth_probs[:, 1:2]
    
    fused_alert = w_base * base_alert + w_eye * eye_alert + w_mouth * mouth_alert
    fused_drowsy = 1.0 - fused_alert
    return torch.cat([fused_drowsy, fused_alert], dim=1)

print("\n" + "="*60)
print("TESTING LATE FUSION")
print("="*60)

model.eval()
all_probs = {'d': [], 'e': [], 'm': []}
all_labels_d = []

with torch.no_grad():
    for x, y_d, _, _, lm in tqdm(test_loader, desc="Computing probs"):
        x, lm = x.to(device), lm.to(device)
        logits_d, logits_e, logits_m = model(x, lm)
        
        all_probs['d'].append(F.softmax(logits_d, dim=1).cpu())
        all_probs['e'].append(F.softmax(logits_e, dim=1).cpu())
        all_probs['m'].append(F.softmax(logits_m, dim=1).cpu())
        all_labels_d.append(y_d)

drowsy_probs = torch.cat(all_probs['d'])
eye_probs = torch.cat(all_probs['e'])
mouth_probs = torch.cat(all_probs['m'])
labels_d = torch.cat(all_labels_d).numpy()

fused_probs = late_fusion_predict(drowsy_probs, eye_probs, mouth_probs,
                                  CFG["fusion_w_base"], CFG["fusion_w_eye"], CFG["fusion_w_mouth"])
fused_preds = fused_probs.argmax(1).numpy()
test_acc_fused = accuracy_score(labels_d, fused_preds)

print(f"\nBase Model Accuracy:  {test_acc_base:.4f}")
print(f"Fused Model Accuracy: {test_acc_fused:.4f}")
print(f"Improvement: +{(test_acc_fused - test_acc_base):.4f} ({(test_acc_fused/test_acc_base - 1)*100:.2f}%)\n")

print("=== Fused Model Classification Report ===")
print(classification_report(labels_d, fused_preds, target_names=['Drowsy', 'Alert'], digits=4))

# Save Result

In [ ]:
# Save model
torch.save(best_state, MODEL2_DIR / "model2_best.pth")

# Save results
cm_base = confusion_matrix(labels_d, test_preds['d'])
cm_fused = confusion_matrix(labels_d, fused_preds)

results = {
    "config": CFG,
    "best_epoch": best_epoch,
    "best_val_acc": float(best_val_acc),
    "test_acc_base": float(test_acc_base),
    "test_acc_fused": float(test_acc_fused),
    "improvement": float(test_acc_fused - test_acc_base),
    "confusion_matrix_base": cm_base.tolist(),
    "confusion_matrix_fused": cm_fused.tolist()
}

write_json(results, MODEL2_DIR / "results.json")

# Plot
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Training curves
axes[0].plot(history['train_loss'], label='Train', marker='o')
axes[0].plot(history['val_loss'], label='Val', marker='s')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(True)

axes[1].plot(history['train_acc'], label='Train', marker='o')
axes[1].plot(history['val_acc'], label='Val', marker='s')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(True)

# Confusion matrices
axes[2].imshow(cm_fused, cmap='Greens')
axes[2].set_title(f'Fused Model CM\nAcc: {test_acc_fused:.4f}')
axes[2].set_xticks([0,1]); axes[2].set_yticks([0,1])
axes[2].set_xticklabels(['Drowsy', 'Alert'])
axes[2].set_yticklabels(['Drowsy', 'Alert'])
for i in range(2):
    for j in range(2):
        axes[2].text(j, i, str(cm_fused[i, j]), ha='center', va='center')

plt.tight_layout()
plt.savefig(MODEL2_DIR / "results.png", dpi=150)
plt.show()

print(f"\n{'='*60}")
print("COMPLETE!")
print(f"{'='*60}")
print(f"Model saved: {MODEL2_DIR / 'model2_best.pth'}")
print(f"Results saved: {MODEL2_DIR / 'results.json'}")
print(f"{'='*60}")